In [ ]:
from openai imoprt OpenAI

client = OpenAI(
    api_key = ""
)
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {
            "role": "user",
            "content": "Hello world"
        }
    ]
)
print(response.choices[0].message.content)

In [ ]:
client = OpenAI(api_key = "")

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {
            "role": "user",
            "content": "Write down five tree with their scientific name in json format"
        }
    ],
    response_format = {
        "type": "json_object"
    }
)
print(response.choices[0].message.content)

# Handling error

In [ ]:
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Use the try statement
try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[message]
    )
    # Print the response
    print(response.choices[0].message.content)
# Use the except statement
except AuthenticationError:
    print("Please double check your authentication key and try again, the one provided is not valid.")

# Batching

In [ ]:
# Import the tenacity library
from tenacity import retry, wait_random_exponential, stop_after_attempt

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Add the appropriate parameters to the decorator
@retry(wait=wait_random_exponential(min=5, max=40), stop=stop_after_attempt(4))
def get_response(model, message):
    response = client.chat.completions.create(
        model=model,
        messages=[message]
    )
    return response.choices[0].message.content
print(get_response("gpt-4o-mini", {"role": "user", "content": "List ten holiday destinations."}))


# Sending messages in batching to reduce API request to avoid rate limit
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

messages = []
# Provide a system message and user messages to send the batch
messages.append({"role": "system", "content": "Convert each of the measurements from kilometers to miles and present the results in a table containing both the original and converted measurements."})
# Append measurements to the message
[messages.append({"role": "user", "content": str(i)}) for i in measurements]

response = get_response(messages)
print(response)


# Setting token limit
client = OpenAI(api_key="<OPENAI_API_TOKEN>")
input_message = {"role": "user", "content": "I'd like to buy a shirt and a jacket. Can you suggest two color pairings for these items?"}

# Use tiktoken to create the encoding for your model
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
# Check for the number of tokens
num_tokens = len(encoding.encode(input_message["content"]))

# Run the chat completions function and print the response
if num_tokens <= 100:
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[input_message])
    print(response.choices[0].message.content)
else:
    print("Message exceeds token limit")

# Extracting structured data from text

In [ ]:
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

response= client.chat.completions.create(
    model="gpt-4o-mini",
    # Add the message
    messages=message_listing,
    # Add your function definition
    tools=function_definition
)

# Print the response
print(print(response.choices[0].message.tool_calls[0].function.arguments))


# Building a function dictionary
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Define the function parameter type
function_definition[0]['function']['parameters']['type'] = "object"

# Define the function properties
# Define the function properties
function_definition[0]['function']['parameters']['properties'] = {
    "title": {"type": "string", "description": "The title of the research paper"},
    "year": {"type": "string", "description": "The year the paper was published"}
}

response = get_response(messages, function_definition)
print(response)


# Extracting the response
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

response = get_response(messages, function_definition)

# Define the function to extract the data dictionary
def extract_dictionary(response):
    return response.choices[0].message.tool_calls[0].function.arguments

# Print the data dictionary
print(extract_dictionary(response))


# **Working with multiple functions**

In [ ]:
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

response= client.chat.completions.create(
    model=model,
    messages=messages,
    # Add the function definition
    tools=function_definition,
    # Specify the function to be called for the response
    tool_choice={'type': 'function', 'function': {'name': 'extract_review_info'}}
)

# Print the response
print(response.choices[0].message.tool_calls[0].function.arguments)

# Avoiding inconsistent responses
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Modify the messages
messages.append({"role": "system", "content": "Don't make assumptions about what values to plug into functions. Don't make up values to fill the response with."})

response = get_response(messages, function_definition)

print(response)

# **Calling external APIs**

In [1]:
import requests
import json

url = "https://api.artic.edu/api/v1/afterworks/search"
querystring = {"q": keyword}
response = requests.request("GET", url, params=querystring)

def get_artwork(keyword):
  url = "https://api.artic.edu/api/v1/afterworks/search"
  querystring = {"q": keyword}
  reponse = requests.request("GET", url, params=querystring)
  return response.text


response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {
            "role": "system",
            "content": "You are an AI assistant expert in history of art."
        }
    ]
)

if response.choices[0].finish_reason == 'tool_calls':
  function_call = response.choices[0].message.tool_calls[0].function
  if function_call.name == 'get_artwork':
    artwork_keyword = json.loads(function_call.arguments)["artwork keyword"]
    artwork = get_artwork(artwork_keyword)

    if artwork:
      print(f"Here are some recommendations: {[i['title'] for i in json.loads(artwork)['data']]}")

    else:
      print("Apologies, I coudn't make any recommendations based on the request")

  else:
    print("Apologies, I coudn't find any artwork")

else:
  print("Apologies, I coudn't understand your request")

SyntaxError: invalid syntax (2573888267.py, line 18)

In [ ]:
# calling an external API

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Call the Chat Completions endpoint
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a currency exchange assistant. Extract the corresponding currency code based on the user input."},
        {"role": "user", "content": "I'd like to know the current exchange rates for the Euro."}
    ],
    tools=function_definition
)

print_response(response)


# Handling the response with external API calls
# Check that the response has been produced using function calling
if response.choices[0].finish_reason == 'tool_calls':
    # Extract the function
    function_call = response.choices[0].message.tool_calls[0].function
    print(function_call)
else:
    print("I am sorry, but I could not understand your request.")


if response.choices[0].finish_reason=='tool_calls':
    function_call = response.choices[0].message.tool_calls[0].function
    # Check function name
    if function_call.name == "get_exchange_rate":
        # Extract currency code
        code = json.loads(function_call.arguments)["currency_code"]
        exchange_info = get_exchange_rate(code)
        print(exchange_info)
    else:
        print("Apologies, I couldn't find the requested currency.")
else:
    print("I am sorry, but I could not understand your request.")

